# Estatísticas de Variáveis Categóricas

---

## Objetivo

Produzir **frequências, percentuais e moda** para as principais variáveis categóricas das bases analisadas, com tabelas e gráficos.

### Métricas aplicadas a variáveis categóricas

| Métrica | Definição |
|:---|:---|
| **Frequência absoluta** | Contagem de registros em cada categoria |
| **Frequência relativa (%)** | Proporção de cada categoria sobre o total válido |
| **Moda** | Categoria mais frequente — o valor que aparece mais vezes |

> Para variáveis com muitas categorias, apresentamos apenas as **top-N** mais frequentes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

AZUL_ESCURO  = '#1a3a5c'
AZUL_MEDIO   = '#2d6a9f'
AZUL_CLARO   = '#4a90c4'
LARANJA      = '#e07b39'

BASE_PATH = '../data/raw/'
pesquisas    = pd.read_csv(BASE_PATH + 'pesquisas.csv',    sep=';')
empresas     = pd.read_csv(BASE_PATH + 'empresas.csv',     sep=';')
clientes     = pd.read_csv(BASE_PATH + 'clientes.csv',     sep=';')
respondentes = pd.read_csv(BASE_PATH + 'respondentes.csv', sep=';')
questoes     = pd.read_csv(BASE_PATH + 'questoes.csv',     sep=';')

def freq_table(series, top_n=None, dropna=True):
    """Retorna DataFrame com frequência absoluta e relativa."""
    counts = series.value_counts(dropna=dropna)
    if top_n:
        counts = counts.head(top_n)
    total = counts.sum()
    return pd.DataFrame({
        'Categoria': counts.index,
        'Frequência': counts.values,
        'Percentual (%)': (counts.values / total * 100).round(1),
    })

print('Setup concluído.')

---

## 1. `pesquisas.status` — Etapa atual das pesquisas


In [ ]:
tb_status = freq_table(pesquisas['status'])
moda_status = pesquisas['status'].mode()[0]

print('=== pesquisas.status ===')
print(tb_status.to_string(index=False))
print(f'\nModa: "{moda_status}" ({tb_status.set_index("Categoria").loc[moda_status, "Percentual (%)"]:.1f}% dos registros)')

fig, ax = plt.subplots(figsize=(8, 4))
cores = [AZUL_ESCURO if c == moda_status else AZUL_CLARO for c in tb_status['Categoria']]
bars = ax.barh(tb_status['Categoria'], tb_status['Frequência'], color=cores, height=0.65)
for b, (freq, pct) in zip(bars, zip(tb_status['Frequência'], tb_status['Percentual (%)'])):
    ax.text(b.get_width() + 1, b.get_y() + b.get_height() / 2,
            f'{freq} ({pct}%)', va='center', fontsize=9)
ax.set_xlabel('Número de pesquisas')
ax.set_title('Distribuição de pesquisas por status\n(barra escura = moda)', fontweight='bold')
ax.set_xlim(0, tb_status['Frequência'].max() * 1.3)
plt.tight_layout()
plt.savefig('fig_04_status_pesquisas.png', bbox_inches='tight')
plt.show()

**Interpretação:** A esmagadora maioria das pesquisas (≈90,7%) está com status **"Finalizada"**, o que indica que as bases recebidas representam principalmente o histórico, e não o fluxo em andamento. O dashboard D01 deve filtrar por status para separar o acervo histórico do pipeline operacional ativo.

---

## 2. `pesquisas.tipo` — Modalidade das pesquisas


In [ ]:
tb_tipo = freq_table(pesquisas['tipo'])
moda_tipo = pesquisas['tipo'].mode()[0]

print('=== pesquisas.tipo ===')
print(tb_tipo.to_string(index=False))
print(f'\nModa: "{moda_tipo}"')

fig, ax = plt.subplots(figsize=(9, 4))
cores = [AZUL_ESCURO if c == moda_tipo else AZUL_CLARO for c in tb_tipo['Categoria']]
bars = ax.barh(tb_tipo['Categoria'], tb_tipo['Frequência'], color=cores, height=0.65)
for b, (freq, pct) in zip(bars, zip(tb_tipo['Frequência'], tb_tipo['Percentual (%)'])):
    ax.text(b.get_width() + 0.5, b.get_y() + b.get_height() / 2,
            f'{freq} ({pct}%)', va='center', fontsize=9)
ax.set_xlabel('Número de pesquisas')
ax.set_title('Distribuição por tipo de pesquisa', fontweight='bold')
ax.set_xlim(0, tb_tipo['Frequência'].max() * 1.35)
plt.tight_layout()
plt.savefig('fig_05_tipo_pesquisas.png', bbox_inches='tight')
plt.show()

**Interpretação:** "Solicitação de associado" é o tipo mais comum (46,4%), seguido de "Pesquisa fixa anual" (22,9%). Isso significa que a maior parte da carga de trabalho da Central vem de demandas pontuais dos associados, não de pesquisas padronizadas — dado importante para dimensionamento da equipe.

---

## 3. `pesquisas.area` — Área temática


In [ ]:
tb_area = freq_table(pesquisas['area'])
moda_area = pesquisas['area'].mode()[0]

print('=== pesquisas.area ===')
print(tb_area.to_string(index=False))
print(f'\nModa: "{moda_area}"')

# Gráfico de pizza para área (número razoável de categorias)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras horizontais
ax = axes[0]
cores = [AZUL_ESCURO if c == moda_area else AZUL_CLARO for c in tb_area['Categoria']]
bars = ax.barh(tb_area['Categoria'], tb_area['Frequência'], color=cores, height=0.65)
for b, (freq, pct) in zip(bars, zip(tb_area['Frequência'], tb_area['Percentual (%)'])):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2,
            f'{freq} ({pct}%)', va='center', fontsize=8.5)
ax.set_xlabel('Número de pesquisas')
ax.set_title('Pesquisas por área temática', fontweight='bold')
ax.set_xlim(0, tb_area['Frequência'].max() * 1.4)

# Agrupar as menores
ax2 = axes[1]
top5 = tb_area.head(5).copy()
resto = pd.DataFrame([{'Categoria': 'Outras áreas', 'Frequência': tb_area.iloc[5:]['Frequência'].sum(),
                        'Percentual (%)': tb_area.iloc[5:]['Percentual (%)'].sum()}])
pizza_data = pd.concat([top5, resto], ignore_index=True)
cores_pizza = [AZUL_ESCURO, AZUL_MEDIO, AZUL_CLARO, '#6ab0d4', '#99c9e4', '#cccccc']
ax2.pie(pizza_data['Frequência'], labels=pizza_data['Categoria'],
        autopct='%1.1f%%', colors=cores_pizza, startangle=90,
        wedgeprops={'linewidth': 0.5, 'edgecolor': 'white'})
ax2.set_title('Top 5 áreas + Outras', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_06_area_pesquisas.png', bbox_inches='tight')
plt.show()

**Interpretação:** "Sindical Trabalhista e RH" domina com 41,5% das pesquisas, quase o dobro de "Assuntos Regulatórios" (24,0%). As últimas 6 áreas somadas representam apenas 6,6% — são áreas de demanda esporádica que podem ter comportamento diferente de adesão.

---

## 4. `pesquisas.complexidade` — Nível de dificuldade (ordinal)


In [ ]:
# Variável ordinal: fixar a ordem correta
ordem_complexidade = ['baixa', 'média', 'alta']
tb_complexidade = (
    pesquisas['complexidade']
    .value_counts()
    .reindex(ordem_complexidade)
    .reset_index()
)
tb_complexidade.columns = ['Categoria', 'Frequência']
total_valido = tb_complexidade['Frequência'].sum()
tb_complexidade['Percentual (%)'] = (tb_complexidade['Frequência'] / total_valido * 100).round(1)

nulos = pesquisas['complexidade'].isnull().sum()
moda_complexidade = pesquisas['complexidade'].mode()[0]

print('=== pesquisas.complexidade (ordinal: baixa < média < alta) ===')
print(f'  Registros preenchidos: {total_valido} / {len(pesquisas)} ({total_valido/len(pesquisas)*100:.1f}%)')
print(f'  Registros ausentes:    {nulos} ({nulos/len(pesquisas)*100:.1f}%)')
print()
print(tb_complexidade.to_string(index=False))
print(f'\nModa: "{moda_complexidade}"')
print('\n⚠ Atenção: apenas 24,6% dos registros têm complexidade preenchida.')
print('  Os percentuais acima são calculados SOMENTE sobre os registros preenchidos.')

fig, ax = plt.subplots(figsize=(7, 3.5))
cores_ord = [AZUL_CLARO, AZUL_MEDIO, AZUL_ESCURO]  # gradiente escurece com complexidade
bars = ax.bar(tb_complexidade['Categoria'], tb_complexidade['Frequência'],
              color=cores_ord, width=0.55)
for b, (freq, pct) in zip(bars, zip(tb_complexidade['Frequência'], tb_complexidade['Percentual (%)'])):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3,
            f'{freq}\n({pct}%)', ha='center', fontsize=9)
ax.set_ylabel('Número de pesquisas')
ax.set_title(f'Distribuição por complexidade (n={total_valido} registros preenchidos / {len(pesquisas)} total)',
             fontweight='bold')
ax.set_ylim(0, tb_complexidade['Frequência'].max() * 1.3)
plt.tight_layout()
plt.savefig('fig_07_complexidade.png', bbox_inches='tight')
plt.show()

---

## 5. `pesquisas.tipo de resposta` — Nível do respondente


In [ ]:
tb_tresp = freq_table(pesquisas['tipo de resposta'])
print('=== pesquisas.tipo de resposta ===')
print(tb_tresp.to_string(index=False))
print(f'\nModa: "{pesquisas["tipo de resposta"].mode()[0]}"')

**Interpretação:** 93,4% das pesquisas são respondidas no nível **empresa** (uma resposta por organização), enquanto apenas 6,6% coletam respostas de **indivíduo**. Isso afeta diretamente como os KPIs de adesão devem ser calculados — a unidade de contagem muda conforme o tipo.

---

## 6. `empresas.Nacionalidade` e `empresas.Tipo de Associado`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, titulo in [
    (axes[0], 'Nacionalidade', 'Nacionalidade das empresas'),
    (axes[1], 'Tipo de Associado', 'Tipo de associado'),
]:
    tb = freq_table(empresas[col])
    moda = empresas[col].mode()[0]
    print(f'=== empresas.{col} ===')
    print(tb.to_string(index=False))
    print(f'Moda: "{moda}"')
    print()

    cores = [AZUL_ESCURO if c == moda else AZUL_CLARO for c in tb['Categoria']]
    bars = ax.bar(tb['Categoria'], tb['Frequência'], color=cores, width=0.5)
    for b, (freq, pct) in zip(bars, zip(tb['Frequência'], tb['Percentual (%)'])):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1,
                f'{freq}\n({pct}%)', ha='center', fontsize=8.5)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('Empresas')
    ax.set_ylim(0, tb['Frequência'].max() * 1.3)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('fig_08_empresas_categoricas.png', bbox_inches='tight')
plt.show()

---

## 7. `clientes.cargo` e `clientes.departamento`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, top_n, titulo in [
    (axes[0], 'cargo', 8, 'Top 8 cargos (clientes)'),
    (axes[1], 'departamento', 10, 'Top 10 departamentos (clientes)'),
]:
    tb = freq_table(clientes[col], top_n=top_n)
    moda = clientes[col].mode()[0]
    print(f'=== clientes.{col} (top {top_n}) ===')
    print(tb.to_string(index=False))
    print(f'Moda: "{moda}"')
    print()

    cores = [AZUL_ESCURO if c == moda else AZUL_CLARO for c in tb['Categoria']]
    bars = ax.barh(tb['Categoria'][::-1], tb['Frequência'][::-1], color=cores[::-1], height=0.65)
    for b, (freq, pct) in zip(bars, zip(tb['Frequência'][::-1], tb['Percentual (%)'][::-1])):
        ax.text(b.get_width() + 5, b.get_y() + b.get_height() / 2,
                f'{freq} ({pct}%)', va='center', fontsize=8)
    ax.set_xlabel('Clientes cadastrados')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlim(0, tb['Frequência'].max() * 1.4)

plt.tight_layout()
plt.savefig('fig_09_clientes_categoricas.png', bbox_inches='tight')
plt.show()

---

## 8. `questoes.tipo` — Tipo de pergunta


In [ ]:
tb_qtipo = freq_table(questoes['tipo'])
moda_qtipo = questoes['tipo'].mode()[0]

print('=== questoes.tipo ===')
print(tb_qtipo.to_string(index=False))
print(f'\nModa: "{moda_qtipo}"')
print()
print('Implicação para respostas.csv:')
print('  - multipla_escolha e numerica → 1 linha por respondente/questão')
print('  - caixas_de_selecao → N linhas (uma por alternativa marcada)')
print('  - matriz_avaliacao → N×M linhas (linhas × colunas da matriz)')
print('  ⚠ COUNT(*) em respostas.csv NÃO é igual a número de respondentes.')

fig, ax = plt.subplots(figsize=(8, 4))
cores = [AZUL_ESCURO if c == moda_qtipo else AZUL_CLARO for c in tb_qtipo['Categoria']]
bars = ax.barh(tb_qtipo['Categoria'], tb_qtipo['Frequência'], color=cores, height=0.6)
for b, (freq, pct) in zip(bars, zip(tb_qtipo['Frequência'], tb_qtipo['Percentual (%)'])):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2,
            f'{freq} ({pct}%)', va='center', fontsize=9)
ax.set_xlabel('Número de questões')
ax.set_title('Distribuição de questões por tipo', fontweight='bold')
ax.set_xlim(0, tb_qtipo['Frequência'].max() * 1.4)
plt.tight_layout()
plt.savefig('fig_10_tipo_questoes.png', bbox_inches='tight')
plt.show()

---

## 9. Painel-resumo das modas


In [ ]:
resumo_modas = [
    ('pesquisas.status',            pesquisas['status'].mode()[0],
     f"{pesquisas['status'].value_counts().iloc[0]/len(pesquisas)*100:.1f}%"),
    ('pesquisas.tipo',              pesquisas['tipo'].mode()[0],
     f"{pesquisas['tipo'].value_counts().iloc[0]/len(pesquisas)*100:.1f}%"),
    ('pesquisas.area',              pesquisas['area'].mode()[0],
     f"{pesquisas['area'].value_counts().iloc[0]/len(pesquisas)*100:.1f}%"),
    ('pesquisas.complexidade',      pesquisas['complexidade'].mode()[0],
     f"{pesquisas['complexidade'].value_counts().iloc[0]/pesquisas['complexidade'].notna().sum()*100:.1f}% (válidos)"),
    ('pesquisas.tipo de resposta',  pesquisas['tipo de resposta'].mode()[0],
     f"{pesquisas['tipo de resposta'].value_counts().iloc[0]/len(pesquisas)*100:.1f}%"),
    ('empresas.Nacionalidade',      empresas['Nacionalidade'].mode()[0],
     f"{empresas['Nacionalidade'].value_counts().iloc[0]/empresas['Nacionalidade'].notna().sum()*100:.1f}% (válidos)"),
    ('empresas.Tipo de Associado',  empresas['Tipo de Associado'].mode()[0],
     f"{empresas['Tipo de Associado'].value_counts().iloc[0]/empresas['Tipo de Associado'].notna().sum()*100:.1f}% (válidos)"),
    ('clientes.cargo',              clientes['cargo'].mode()[0],
     f"{clientes['cargo'].value_counts().iloc[0]/clientes['cargo'].notna().sum()*100:.1f}% (válidos)"),
    ('clientes.departamento',       clientes['departamento'].mode()[0],
     f"{clientes['departamento'].value_counts().iloc[0]/clientes['departamento'].notna().sum()*100:.1f}% (válidos)"),
    ('questoes.tipo',               questoes['tipo'].mode()[0],
     f"{questoes['tipo'].value_counts().iloc[0]/len(questoes)*100:.1f}%"),
]

df_modas = pd.DataFrame(resumo_modas, columns=['Variável', 'Moda', 'Freq. relativa'])
print('=== Resumo das modas ===')
print(df_modas.to_string(index=False))

## Conclusões

1. **Status:** 90,7% das pesquisas estão "Finalizadas" — os dashboards precisam de filtros claros para separar acervo histórico de pipeline ativo.

2. **Área:** RH domina (41,5%), mas há 13 áreas distintas — o dashboard D02 deve permitir drill-down por área para revelar diferenças de engajamento.

3. **Complexidade:** a moda é "alta" entre os registros preenchidos, mas 75% dos registros não têm esse campo — a análise de tempo por complexidade fica comprometida até o backfill.

4. **Cargo/Departamento:** Gerente e Analista respondem por >50% dos clientes; RH é o departamento mais representado. O indicador I10 (engajamento por público) tem base sólida nessas variáveis.

5. **Tipo de questão:** `multipla_escolha` e `caixas_de_selecao` juntos representam 88% das perguntas — é fundamental tratar esses dois tipos de forma diferente ao contar respostas em `respostas.csv`.
